# Processing gnomAD and ClinVar data

In [1]:
import os
import subprocess
import polars as pl
import pandas as pd
import re

# Set strict error handling
def run_command(cmd):
    result = subprocess.run(cmd, shell=True, check=True, text=True)
    return result

## 1. Loading data

### 1.1 VarChAMP allele collection

Note: here we loaded all the allele collection produced by VarChAMP so far, but not all of them were necessarily profiled by the three assays (DUAL-IPA, Imaging or PPI) in this manuscript.

#### Sequence confirmation results

In [2]:
varchamp_seq_code = pl.read_csv("../1_inputs/seq_confirmation_results/sequence_confirmation_class_code.tsv", separator="\t")
with pl.Config(fmt_str_lengths=150):
    display(varchamp_seq_code)

code,description
i64,str
1,"""perfectly validated"""
2,"""partially validated, >= 50% coverage"""
3,"""wild type"""
4,"""partial wild type, >= 50% coverage"""
5,"""target and off-target mutation, >= 50% coverage"""
6,"""off-target mutation, >= 50% coverage"""
7,"""truncated, < 50% coverage"""
99,"""no reads"""


In [3]:
def map_chr(num):
    if num == 23:
        return "X"
    elif num == 24:
        return "Y"
    elif num == 12920:
        return "MT"
    else:
        return str(num)


varchamp_seq_df = pl.read_csv("../1_inputs/seq_confirmation_results/VarChampSeqConfirmationResult.tsv", separator="\t")
## clean up data
varchamp_seq_df = varchamp_seq_df.with_columns(
    pl.concat_str([pl.col("symbol"), pl.col("aa_change")], separator="_").alias("gene_variant")
).with_columns(
    pl.col("spdi").str.extract(r"NC_0*([0-9]+)", 1).cast(pl.Int32).alias("chr_num"),
    pl.col("spdi").str.split(":").list.get(1).str.strip_chars().cast(pl.Int64).alias("nuc_loc"),
    pl.col("spdi").str.split(":").list.get(2).str.strip_chars().alias("ref_allele"),
    pl.col("spdi").str.split(":").list.get(3).str.strip_chars().alias("alt_allele"),
).with_columns(
    pl.col("chr_num")
    .map_elements(map_chr, return_dtype=pl.String)
    .str.strip_chars()
    .alias("Chrom")
).rename(
    {
        "orf_id_wt": "orf_id",
        "mutation_id_old": "mut_id"
    }
)

# Convert columns to integer (NULL -> None)
columns_to_cast = [
    "entry_sequence_confirmation_class",
    "db_sequence_confirmation_class",
    "n2h_sequence_confirmation_class",
    "n2h_ref_sequence_confirmation_class",
    "dualip_sequence_confirmation_class",
    "dualip_ref_sequence_confirmation_class",
    "mislocalization_sequence_confirmation_class",
    "mislocalization_ref_sequence_confirmation_class"
]

varchamp_seq_df = varchamp_seq_df.with_columns([
    pl.col(col).cast(pl.Int64, strict=False) for col in columns_to_cast
])

# Define masks for allele filtering
entry_good = pl.col("entry_sequence_confirmation_class").is_in([1, 2])
entry_discard = pl.col("entry_sequence_confirmation_class").is_in([3, 4, 5, 6])
entry_null_like = pl.col("entry_sequence_confirmation_class").is_in([7, 99]) | pl.col("entry_sequence_confirmation_class").is_null()

# At least one expression vector is good
any_expression_good = (
    pl.col("db_sequence_confirmation_class").is_in([1, 2]) |
    pl.col("n2h_sequence_confirmation_class").is_in([1, 2]) |
    pl.col("dualip_sequence_confirmation_class").is_in([1, 2]) |
    pl.col("mislocalization_sequence_confirmation_class").is_in([1, 2])
)

# Combine logic
varchamp_seq_df = varchamp_seq_df.filter(
    entry_good | (entry_null_like & any_expression_good)
)

display(varchamp_seq_df)

symbol,ensembl_gene_id,orf_id,mut_id,ccsb_mutation_id,ccsb_allele_id,spdi,nt_change,aa_change,collection,entry_plate_orig,entry_well_orig,entry_plate_conso,entry_well_conso,entry_seq_pool,db_plate,db_well,n2h_plate,n2h_well,dualip_plate,dualip_well,mislocalization_plate,mislocalization_well,entry_sequenced,entry_sequence_confirmation_class,db_sequenced,db_sequence_confirmation_class,n2h_sequenced,n2h_sequence_confirmation_class,n2h_ref_sequence_confirmation_class,dualip_sequenced,dualip_sequence_confirmation_class,dualip_ref_sequence_confirmation_class,mislocalization_sequenced,mislocalization_sequence_confirmation_class,mislocalization_ref_sequence_confirmation_class,gene_variant,chr_num,nuc_loc,ref_allele,alt_allele,Chrom
str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i32,i64,str,str,str
"""GBA1""","""ENSG00000177628""",2,6,"""CCSBVarC000001""","""ALE0000584""","""NC_000001.11:155240033:C:G""","""160G>C""","""Val54Leu""","""RC4""","""RC4_Mut_GDEh1026""","""H01""","""GDEhDisVCh_40054""","""F12""","""2""","""RC4_Mut_GDDh1026""","""H01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1,1,2,0,null,null,0,null,null,0,null,null,"""GBA1_Val54Leu""",1,155240033,"""C""","""G""","""1"""
"""GBA1""","""ENSG00000177628""",2,73,"""CCSBVarC000002""","""ALE00000002""","""NC_000001.11:155238225:G:A""","""670C>T""","""Leu224Phe""","""RC4""","""RC4_Mut_GDEh1026""","""E01""","""GDEhDisVCh_40054""","""C12""","""2""","""RC4_Mut_GDDh1026""","""E01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1,1,1,0,null,null,0,null,null,0,null,null,"""GBA1_Leu224Phe""",1,155238225,"""G""","""A""","""1"""
"""GBA1""","""ENSG00000177628""",2,113,"""CCSBVarC000003""","""ALE00000003""","""NC_000001.11:155237453:C:T""","""887G>A""","""Arg296Gln""","""RC4""","""RC4_Mut_GDEh1026""","""F01""","""GDEhDisVCh_40054""","""D12""","""2""","""RC4_Mut_GDDh1026""","""F01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1,1,7,0,null,null,0,null,null,0,null,null,"""GBA1_Arg296Gln""",1,155237453,"""C""","""T""","""1"""
"""GBA1""","""ENSG00000177628""",2,231,"""CCSBVarC000004""","""ALE00000004""","""NC_000001.11:155235252:A:G""","""1448T>C""","""Leu483Pro""","""RC4""","""RC4_Mut_GDEh1026""","""G01""","""GDEhDisVCh_40054""","""E12""","""2""","""RC4_Mut_GDDh1026""","""G01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1,1,2,0,null,null,0,null,null,0,null,null,"""GBA1_Leu483Pro""",1,155235252,"""A""","""G""","""1"""
"""GBA1""","""ENSG00000177628""",2,213510,"""CCSBVarC003869""","""ALE00003869""","""NC_000001.11:155239934:G:A""","""259C>T""","""Arg87Trp""","""CEGS2""","""CegsMutGDEh1035""","""B03""","""GDEhDisVCh_40054""","""B02""","""2""","""CegsMutGDDh1035""","""B03""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1,1,2,0,null,null,0,null,null,0,null,null,"""GBA1_Arg87Trp""",1,155239934,"""G""","""A""","""1"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SRY""","""ENSG00000184895""",100070152,229149,"""CCSBVarC007069""","""ALE00007067""","""NC_000024.10:2787412:C:T""","""192G>A""","""Met64Ile""","""CEGS2""","""CegsMutGDEh1023""","""D05""","""GDEhDisVCh_40029""","""D09""","""1""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1,0,null,0,null,null,0,null,null,0,null,null,"""SRY_Met64Ile""",24,2787412,"""C""","""T""","""Y"""
"""SRY""","""ENSG00000184895""",100070152,229151,"""CCSBVarC007072""","""ALE00007070""","""NC_000024.10:2787267:C:T""","""337G>A""","""Ala113Thr""","""CEGS2""","""CegsMutGDEh1023""","""F09""","""GDEhDisVCh_40029""","""H09""","""1""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1,0,null,0,null,null,0,null,null,0,null,null,"""SRY_Ala113Thr""",24,2787267,"""C""","""T""","""Y"""
"""SRY""","""ENSG00000184895""",100070152,229152,"""CC

In [4]:
cols_to_keep = [
    "spdi",
    "symbol",
    "gene_variant",
    "orf_id",
    "ccsb_mutation_id",
    "nt_change", #"nt_change_cdna",
    "aa_change",
    # "node_type",
    # "dest_well",
    # "dest_plate",
    # "imaging_well",
    # "imaging_plate_R1",
    # "imaging_plate_R2"
]

cava = pl.read_csv("../1_inputs/seq_confirmation_results/B13_B14_Metadata_update_SPDI.csv")
cava = cava.rename(
    {"gene_allele": "gene_variant"}
).with_columns(
    # regex replace a single trailing underscore
    pl.col("gene_variant").str.replace(r"_$", "", literal=False).alias("gene_variant")
).filter(
    (pl.col("node_type") == "allele")
    & ~pl.col("gene_variant").str.contains("CCM2")
)

# filter for node_type == "allele" and spdi null/NA
missing_spdi_cava = cava.filter(
    (pl.col("node_type") == "allele") &
    pl.col("spdi").is_null()
)
# get the count
print("Number of CAVA alleles that are missing spdi is:", missing_spdi_cava.shape[0]) # these appear to come from RC4 mutant collection

cava_imaging_df = cava.rename(
    {
        "orf_id_wt": "orf_id",
        "nt_change_cdna": "nt_change"
    }
).select(
    cols_to_keep
).with_columns(
    pl.col("spdi").str.extract(r"NC_0*([0-9]+)", 1).cast(pl.Int32).alias("chr_num"),
    pl.col("spdi").str.split(":").list.get(1).str.strip_chars().cast(pl.Int64).alias("nuc_loc"),
    pl.col("spdi").str.split(":").list.get(2).str.strip_chars().alias("ref_allele"),
    pl.col("spdi").str.split(":").list.get(3).str.strip_chars().alias("alt_allele"),
).with_columns(
    pl.col("chr_num")
    .map_elements(map_chr, return_dtype=pl.String)
    .str.strip_chars()
    .alias("Chrom")
)
display(cava_imaging_df)

Number of CAVA alleles that are missing spdi is: 5


spdi,symbol,gene_variant,orf_id,ccsb_mutation_id,nt_change,aa_change,chr_num,nuc_loc,ref_allele,alt_allele,Chrom
str,str,str,i64,str,str,str,i32,i64,str,str,str
"""NC_000023.11:153740155:C:T""","""ABCD1""","""ABCD1_Arg518Trp""",100008637,"""CCSBVarC007022""","""1552C>T""","""Arg518Trp""",23,153740155,"""C""","""T""","""X"""
"""NC_000023.11:153736195:C:G""","""ABCD1""","""ABCD1_Arg389Gly""",100008637,"""CCSBVarC007023""","""1165C>G""","""Arg389Gly""",23,153736195,"""C""","""G""","""X"""
"""NC_000023.11:153740156:G:A""","""ABCD1""","""ABCD1_Arg518Gln""",100008637,"""CCSBVarC007021""","""1553G>A""","""Arg518Gln""",23,153740156,"""G""","""A""","""X"""
"""NC_000003.12:52403242:T:C""","""BAP1""","""BAP1_Ser596Gly""",3760,"""CCSBVarC009473""","""1786A>G""","""Ser596Gly""",3,52403242,"""T""","""C""","""3"""
"""NC_000003.12:52403877:G:T""","""BAP1""","""BAP1_Thr423Lys""",3760,"""CCSBVarC009476""","""1268C>A""","""Thr423Lys""",3,52403877,"""G""","""T""","""3"""
…,…,…,…,…,…,…,…,…,…,…,…
"""NC_000023.11:64921845:A:T""","""ZC4H2""","""ZC4H2_Leu66His""",4320,"""CCSBVarC009188""","""197T>A""","""Leu66His""",23,64921845,"""A""","""T""","""X"""
"""NC_000023.11:64917865:C:T""","""ZC4H2""","""ZC4H2_Arg198Gln""",4320,"""CCSBVarC009191""","""593G>A""","""Arg198Gln""",23,64917865,"""C""","""T""","""X"""
"""NC_000023.11:64976325:C:T""","""ZC4H2""","""ZC4H2_Arg18Lys""",4320,"""CCSBVarC009194""","""53G>A""","""Arg18Lys""",23,64976325,"""C""","""T""","""X"""


In [5]:
varchamp_df = pl.concat(
    [
        varchamp_seq_df, 
        cava_imaging_df,
    ],
    how="diagonal_relaxed",
    rechunk=True
).with_columns([
    # Fix: Use concat_str for the varid string concatenation
    pl.concat_str([
        pl.col("Chrom"),
        pl.lit("-"),
        pl.col("nuc_loc").cast(pl.String),
        pl.lit("-"),
        pl.col("ref_allele"),
        pl.lit("-"),
        pl.col("alt_allele")
    ]).alias("varid")
])
varchamp_df

symbol,ensembl_gene_id,orf_id,mut_id,ccsb_mutation_id,ccsb_allele_id,spdi,nt_change,aa_change,collection,entry_plate_orig,entry_well_orig,entry_plate_conso,entry_well_conso,entry_seq_pool,db_plate,db_well,n2h_plate,n2h_well,dualip_plate,dualip_well,mislocalization_plate,mislocalization_well,entry_sequenced,entry_sequence_confirmation_class,db_sequenced,db_sequence_confirmation_class,n2h_sequenced,n2h_sequence_confirmation_class,n2h_ref_sequence_confirmation_class,dualip_sequenced,dualip_sequence_confirmation_class,dualip_ref_sequence_confirmation_class,mislocalization_sequenced,mislocalization_sequence_confirmation_class,mislocalization_ref_sequence_confirmation_class,gene_variant,chr_num,nuc_loc,ref_allele,alt_allele,Chrom,varid
str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i32,i64,str,str,str,str
"""GBA1""","""ENSG00000177628""",2,6,"""CCSBVarC000001""","""ALE0000584""","""NC_000001.11:155240033:C:G""","""160G>C""","""Val54Leu""","""RC4""","""RC4_Mut_GDEh1026""","""H01""","""GDEhDisVCh_40054""","""F12""","""2""","""RC4_Mut_GDDh1026""","""H01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1,1,2,0,null,null,0,null,null,0,null,null,"""GBA1_Val54Leu""",1,155240033,"""C""","""G""","""1""","""1-155240033-C-G"""
"""GBA1""","""ENSG00000177628""",2,73,"""CCSBVarC000002""","""ALE00000002""","""NC_000001.11:155238225:G:A""","""670C>T""","""Leu224Phe""","""RC4""","""RC4_Mut_GDEh1026""","""E01""","""GDEhDisVCh_40054""","""C12""","""2""","""RC4_Mut_GDDh1026""","""E01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1,1,1,0,null,null,0,null,null,0,null,null,"""GBA1_Leu224Phe""",1,155238225,"""G""","""A""","""1""","""1-155238225-G-A"""
"""GBA1""","""ENSG00000177628""",2,113,"""CCSBVarC000003""","""ALE00000003""","""NC_000001.11:155237453:C:T""","""887G>A""","""Arg296Gln""","""RC4""","""RC4_Mut_GDEh1026""","""F01""","""GDEhDisVCh_40054""","""D12""","""2""","""RC4_Mut_GDDh1026""","""F01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1,1,7,0,null,null,0,null,null,0,null,null,"""GBA1_Arg296Gln""",1,155237453,"""C""","""T""","""1""","""1-155237453-C-T"""
"""GBA1""","""ENSG00000177628""",2,231,"""CCSBVarC000004""","""ALE00000004""","""NC_000001.11:155235252:A:G""","""1448T>C""","""Leu483Pro""","""RC4""","""RC4_Mut_GDEh1026""","""G01""","""GDEhDisVCh_40054""","""E12""","""2""","""RC4_Mut_GDDh1026""","""G01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1,1,2,0,null,null,0,null,null,0,null,null,"""GBA1_Leu483Pro""",1,155235252,"""A""","""G""","""1""","""1-155235252-A-G"""
"""GBA1""","""ENSG00000177628""",2,213510,"""CCSBVarC003869""","""ALE00003869""","""NC_000001.11:155239934:G:A""","""259C>T""","""Arg87Trp""","""CEGS2""","""CegsMutGDEh1035""","""B03""","""GDEhDisVCh_40054""","""B02""","""2""","""CegsMutGDDh1035""","""B03""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1,1,2,0,null,null,0,null,null,0,null,null,"""GBA1_Arg87Trp""",1,155239934,"""G""","""A""","""1""","""1-155239934-G-A"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ZC4H2""",null,4320,null,"""CCSBVarC009188""",null,"""NC_000023.11:64921845:A:T""","""197T>A""","""Leu66His""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""ZC4H2_Leu66His""",23,64921845,"""A""","""T""","""X""","""X-64921845-A-T"""
"""ZC4H2""",null,4320,null,"""CCSBVarC009191""",null,"""NC_000023.11:64917865:C:T""","""593G>A""","""Arg198Gln""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""ZC4H2_Arg198Gln""",23,64917865,"""C""","""T""","""X""","""X-64917865-C-T"""
"""ZC4H2""",null,4320,null,"""CCSBVarC009194""",null,"""NC_000023.11:64976325:C:T""","""53G>A""","""Arg18Lys""",null,null,

### 1.2 ClinVar

ClinVar variants released on 2024-03-01 and the data is accessed on March 13, 2025

Cleaning up ClinVar annotations. For merging the ClinVar with the VarChAMP variants more easily

In [6]:
dest_path = "../1_inputs/raw_inputs/clinvar/variant_summary.txt.gz"
## Define the new processed and cleaned clinvar file
processed_clinvar_parq = "../3_outputs/processed_data/clinvar/clinvar_processed.parquet"

In [4]:
# ## Read CSV with Polars (auto-decompresses .gz)
# clinvar = pl.read_csv(dest_path, separator='\t', infer_schema_length=1000000)
# ## Process data with Polars expressions
# clinvar = (
#     clinvar
#     .filter(pl.col("Assembly") == "GRCh38")
#     .with_columns(
#         pl.col("Name")
#         # Split the string on '.', '(' or ')'
#         .map_elements(lambda s: re.split(r"[.\(\)]", s), pl.List(pl.String))
#         .alias("Name_listed")
#     )
# )
# clinvar = clinvar.with_columns(
#     pl.col("Name_listed").list.get(0).alias("NM"),
#     pl.col("Name_listed").list.get(1).alias("version"), 
#     pl.col("Name_listed").list.get(2).alias("gene"),
#     pl.col("Name_listed").list.get(3).alias("c"),
#     pl.col("Name_listed").list.get(4).alias("clinvar_nt_change"),
#     pl.col("Name_listed").list.get(5).alias("p"),
#     pl.col("Name_listed").list.get(6).alias("clinvar_aa_change")
# ).with_columns(
#         SPDI=pl.col("ChromosomeAccession") + ":" + 
#             pl.col("PositionVCF").cast(str) + ":" +
#             pl.col("ReferenceAlleleVCF") + ":" +
#             pl.col("AlternateAlleleVCF"),
#         RefSeq_mRNA=pl.col("NM") + "." + pl.col("version"),
#         StarStatus=pl.when(pl.col("ReviewStatus").str.contains(r'single|conflicting'))
#             .then(1)
#             .when(pl.col("ReviewStatus").str.contains('multiple'))
#             .then(2)
#             .when(pl.col("ReviewStatus").str.contains('expert'))
#             .then(3)
#             .when(pl.col("ReviewStatus").str.contains('practice'))
#             .then(4)
#             .otherwise(None)
# )

In [4]:
## Create the mapping expression with precise logic for a cleaner ClinVar Annotation
clinvar_mapping = (
    # First check for Conflicting interpretations (highest priority)
    pl.when(
        pl.col("ClinicalSignificance").str.to_lowercase().str.contains("conflicting")
    )
    .then(pl.lit("3_Conflicting"))
    
    # Then check for Uncertain significance (VUS)
    .when(pl.col("ClinicalSignificance").str.to_lowercase().str.contains("uncertain"))
    .then(pl.lit("4_VUS"))
    
    # # Special case for strings containing both "Uncertain significance" and "Pathogenic"
    # .when(
    #     pl.col("clinvar_clnsig").str.to_lowercase().str.contains("uncertain") &
    #     pl.col("clinvar_clnsig").str.to_lowercase().str.contains("pathogenic")
    # )
    # .then(pl.lit("4_VUS"))  # The dictionary says to classify this as VUS
    
    # Check for Pathogenic
    .when(pl.col("ClinicalSignificance").str.to_lowercase().str.contains("pathogenic"))
    .then(pl.lit("1_Pathogenic"))
    
    # Check for Benign
    .when(pl.col("ClinicalSignificance").str.to_lowercase().str.contains("benign"))
    .then(pl.lit("2_Benign"))
    
    # Everything else goes to Others
    .otherwise(pl.lit("5_Others"))
)

# Apply the mapping to create a new column
clinvar = clinvar.with_columns(clinvar_clnsig_clean=clinvar_mapping)

# Create a dataframe with unique mappings to check accuracy
unique_mappings = clinvar.select(
    pl.col("ClinicalSignificance"),
    pl.col("clinvar_clnsig_clean")
).unique()

# Sort by original value for easier verification
unique_mappings = unique_mappings.sort("ClinicalSignificance")

# Print each mapping to verify accuracy
print("Original Value -> Mapped Value")
print("-" * 50)
for row in unique_mappings.rows():
    print(f"{row[0]} -> {row[1]}")

Original Value -> Mapped Value
--------------------------------------------------
- -> 5_Others
Affects -> 5_Others
Affects; risk factor -> 5_Others
Benign -> 2_Benign
Benign/Likely benign -> 2_Benign
Benign/Likely benign; association -> 2_Benign
Benign/Likely benign; drug response -> 2_Benign
Benign/Likely benign; drug response; other -> 2_Benign
Benign/Likely benign; other -> 2_Benign
Benign/Likely benign; other; risk factor -> 2_Benign
Benign/Likely benign; risk factor -> 2_Benign
Benign; Affects -> 2_Benign
Benign; Affects; association; other -> 2_Benign
Benign; association -> 2_Benign
Benign; confers sensitivity -> 2_Benign
Benign; drug response -> 2_Benign
Benign; other -> 2_Benign
Benign; protective -> 2_Benign
Benign; risk factor -> 2_Benign
Conflicting classifications of pathogenicity -> 3_Conflicting
Conflicting classifications of pathogenicity; Affects -> 3_Conflicting
Conflicting classifications of pathogenicity; association -> 3_Conflicting
Conflicting classifications of p

In [5]:
# ## Make sure we only drop columns that actually exist
# columns_to_drop = [col for col in ['NM', 'version', 'gene', 'c', 'p', 'ReferenceAllele', 'AlternateAllele', 'Name_listed'] 
#                   if col in clinvar.columns]
# if columns_to_drop:
#     clinvar = clinvar.drop(columns_to_drop)

# # Now cast to categorical
# clinvar = clinvar.with_columns(
#     pl.col(["Assembly", "ClinicalSignificance", "ReviewStatus", "clinvar_clnsig_clean"]).cast(pl.Categorical)
# )

# # Write to Parquet with ZSTD compression
# clinvar.write_parquet(processed_clinvar_parq, 
#                      compression="zstd", 
#                      statistics=True)

### 1.3 Merge with gnomad

##### Sort out all the vcf files and all their position coordinates for looking up varchamp-related variants

In [7]:
## Generate all the position coordinates for varchamp-related variants for quicker look-up

## Define some constants
VCF_DIR = "../1_inputs/raw_inputs/gnomad"
CHROMOSOMES = [str(c) for c in list(range(1, 23))] + ['X', 'Y']

# VARIANT_LIST = f"../3_outputs/processed_data/gnomad_allele_frequencies/gnomad_variants_list.txt"
# gene_info_pd[["chrom","chr_pos_38","ref","alt"]].to_csv(VARIANT_LIST, sep="\t", header=False, index=False)

In [8]:
# # !conda install bcftools -c conda-forge -y ## install bcftools if you didn't

# OUTPUT_DIR = f"../3_outputs/processed_data/gnomad_allele_frequencies"
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# # Process chromosomes
# for chrom in CHROMOSOMES:
#     vcf_file = f"{VCF_DIR}/gnomad.exomes.v4.1.sites.chr{chrom}.vcf.bgz"
#     csi_file = f"{vcf_file}.csi"
#     output_file = f"{OUTPUT_DIR}/chr{chrom}_allele_frequencies.txt"
    
#     if os.path.exists(vcf_file):
#         print(f"Processing chromosome {chrom}...")
        
#         if not os.path.exists(csi_file):
#             print(f"Indexing {vcf_file}")
#             run_command(f"bcftools index --threads 4 {vcf_file}")
        
#         print(f"Extracting frequencies for {chrom}")
#         view_cmd = f"bcftools view -R {VARIANT_LIST} {vcf_file}"
#         query_cmd = "bcftools query -f '%CHROM\\t%POS\\t%REF\\t%ALT\\t%AF\\n'"
        
#         with open(output_file, 'w') as f:
#             p1 = subprocess.Popen(view_cmd.split(), stdout=subprocess.PIPE)
#             p2 = subprocess.Popen(query_cmd.split(), stdin=p1.stdout, stdout=f)
#             p1.stdout.close()
#             p2.communicate()
            
#         print(f"Saved to {output_file}")
#     else:
#         print(f"Skipping chromosome {chrom} - file not found")

# print("Processing completed")

#### Merge the gnomad AF files from chromosomes

In [9]:
allele_freq_dfs = []
for chrom in CHROMOSOMES:
    try:
        allele_freq = pl.read_csv(f"../3_outputs/processed_data/gnomad_allele_frequencies/chr{chrom}_allele_frequencies.txt",
                                separator="\t", has_header=False, 
                                new_columns=["chr","chr_pos_38","ref","alt","allele_freq"])
        allele_freq = allele_freq.with_columns(
            pl.col("chr").str.replace(r"\'", "").str.replace(r"\'", "").str.replace("chr", "")
        ).drop_nulls()
        # allele_freq = allele_freq.filter(~pl.all_horizontal(pl.all().is_null()))
        # print(allele_freq)
        allele_freq_dfs.append(allele_freq)
    except:
        continue

allele_freq_all = pl.concat(allele_freq_dfs)
allele_freq_gnomad = allele_freq_all.with_columns(
    pl.concat_str([pl.col("chr"),
                   pl.lit('-'),
                   pl.col("chr_pos_38").cast(pl.String),
                   pl.lit('-'),
                   pl.col("ref"),
                   pl.lit('-'),
                   pl.col("alt")
                   ]).alias("varid"),
    pl.col("allele_freq").alias("gnomad_af")
).drop(pl.col("allele_freq"))#.to_pandas()

allele_freq_gnomad

chr,chr_pos_38,ref,alt,varid,gnomad_af
str,i64,str,str,str,f64
"""1""",2028259,"""C""","""A""","""1-2028259-C-A""",6.8500e-7
"""1""",2028259,"""C""","""T""","""1-2028259-C-T""",0.001409
"""1""",2028260,"""G""","""A""","""1-2028260-G-A""",0.0217404
"""1""",5907193,"""CG""","""C""","""1-5907193-CG-C""",0.0
"""1""",5907194,"""G""","""A""","""1-5907194-G-A""",0.000013
…,…,…,…,…,…
"""X""",154863124,"""C""","""T""","""X-154863124-C-T""",0.000011
"""X""",154863125,"""G""","""A""","""X-154863125-G-A""",0.000004
"""X""",154863151,"""C""","""T""","""X-154863151-C-T""",0.000004


## 2. Merge clinvar and gnomad to varchamp alleles

### 2.1 Merge with ClinVar

In [10]:
processed_clinvar_parq = "../3_outputs/processed_data/clinvar/clinvar_processed.parquet"
clinvar = pl.scan_parquet(processed_clinvar_parq)
clinvar.collect().shape
# clinvar.collect().head()

(3390885, 47)

In [11]:
varchamp_clinvar = varchamp_df.lazy().join(
                        clinvar,
                        left_on="spdi",
                        right_on="SPDI",
                        how="left"
                    )

#### Report the number of alleles missing ClinVar annotations

In [12]:
# gene_info_clinvar = pl.scan_parquet(gene_info_clinvar_parq)
varchamp_clinvar.filter((~pl.col("spdi").is_null())&(pl.col("#AlleleID").is_null())).collect()

symbol,ensembl_gene_id,orf_id,mut_id,ccsb_mutation_id,ccsb_allele_id,spdi,nt_change,aa_change,collection,entry_plate_orig,entry_well_orig,entry_plate_conso,entry_well_conso,entry_seq_pool,db_plate,db_well,n2h_plate,n2h_well,dualip_plate,dualip_well,mislocalization_plate,mislocalization_well,entry_sequenced,entry_sequence_confirmation_class,db_sequenced,db_sequence_confirmation_class,n2h_sequenced,n2h_sequence_confirmation_class,n2h_ref_sequence_confirmation_class,dualip_sequenced,dualip_sequence_confirmation_class,dualip_ref_sequence_confirmation_class,mislocalization_sequenced,mislocalization_sequence_confirmation_class,mislocalization_ref_sequence_confirmation_class,gene_variant,…,RS# (dbSNP),nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,clinvar_nt_change,clinvar_aa_change,RefSeq_mRNA,StarStatus,clinvar_clnsig_clean
str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,…,i64,str,str,str,str,str,str,cat,str,str,i64,i64,str,cat,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,cat
"""GBA1""","""ENSG00000177628""",2,6,"""CCSBVarC000001""","""ALE0000584""","""NC_000001.11:155240033:C:G""","""160G>C""","""Val54Leu""","""RC4""","""RC4_Mut_GDEh1026""","""H01""","""GDEhDisVCh_40054""","""F12""","""2""","""RC4_Mut_GDDh1026""","""H01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1,1,2,0,null,null,0,null,null,0,null,null,"""GBA1_Val54Leu""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""GBA1""","""ENSG00000177628""",2,73,"""CCSBVarC000002""","""ALE00000002""","""NC_000001.11:155238225:G:A""","""670C>T""","""Leu224Phe""","""RC4""","""RC4_Mut_GDEh1026""","""E01""","""GDEhDisVCh_40054""","""C12""","""2""","""RC4_Mut_GDDh1026""","""E01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1,1,1,0,null,null,0,null,null,0,null,null,"""GBA1_Leu224Phe""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""ACTA2""","""ENSG00000107796""",167,390,"""CCSBVarC000012""","""ALE00000012""","""NC_000010.11:88938177:T:C""","""874A>G""","""Arg292Gly""","""RC4""","""RC4_Mut_GDEh1018""","""D12""","""GDEhDisVCh_40054""","""B12""","""2""","""RC4_Mut_GDDh1018""","""D12""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1,1,1,0,null,null,0,null,null,0,null,null,"""ACTA2_Arg292Gly""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""FOXA3""","""ENSG00000170608""",187,393,"""CCSBVarC000013""","""ALE00000013""","""NC_000019.10:45872276:G:A""","""271G>A""","""Gly91Arg""","""RC4""","""RC4_Mut_GDEh1002""","""F09""","""GDEhDisVCh_40008""","""G09""","""2""","""RC4_Mut_GDDh1002""","""F09""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1,1,1,0,null,null,0,null,null,0,null,null,"""FOXA3_Gly91Arg""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""ABHD5""","""ENSG00000011198""",286,782,"""CCSBVarC000031""","""ALE

Check why missing, and it seems like some alleles just weren't collected by ClinVar yet

In [13]:
varchamp_clinvar.filter(pl.col("symbol")=="GBA1").sort(by="clinvar_aa_change").collect()

symbol,ensembl_gene_id,orf_id,mut_id,ccsb_mutation_id,ccsb_allele_id,spdi,nt_change,aa_change,collection,entry_plate_orig,entry_well_orig,entry_plate_conso,entry_well_conso,entry_seq_pool,db_plate,db_well,n2h_plate,n2h_well,dualip_plate,dualip_well,mislocalization_plate,mislocalization_well,entry_sequenced,entry_sequence_confirmation_class,db_sequenced,db_sequence_confirmation_class,n2h_sequenced,n2h_sequence_confirmation_class,n2h_ref_sequence_confirmation_class,dualip_sequenced,dualip_sequence_confirmation_class,dualip_ref_sequence_confirmation_class,mislocalization_sequenced,mislocalization_sequence_confirmation_class,mislocalization_ref_sequence_confirmation_class,gene_variant,…,RS# (dbSNP),nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,clinvar_nt_change,clinvar_aa_change,RefSeq_mRNA,StarStatus,clinvar_clnsig_clean
str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,…,i64,str,str,str,str,str,str,cat,str,str,i64,i64,str,cat,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,cat
"""GBA1""","""ENSG00000177628""",2,6,"""CCSBVarC000001""","""ALE0000584""","""NC_000001.11:155240033:C:G""","""160G>C""","""Val54Leu""","""RC4""","""RC4_Mut_GDEh1026""","""H01""","""GDEhDisVCh_40054""","""F12""","""2""","""RC4_Mut_GDDh1026""","""H01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1,1,2,0,null,null,0,null,null,0,null,null,"""GBA1_Val54Leu""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""GBA1""","""ENSG00000177628""",2,73,"""CCSBVarC000002""","""ALE00000002""","""NC_000001.11:155238225:G:A""","""670C>T""","""Leu224Phe""","""RC4""","""RC4_Mut_GDEh1026""","""E01""","""GDEhDisVCh_40054""","""C12""","""2""","""RC4_Mut_GDDh1026""","""E01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1,1,1,0,null,null,0,null,null,0,null,null,"""GBA1_Leu224Phe""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""GBA1""","""ENSG00000177628""",2,213560,"""CCSBVarC003889""","""ALE00003889""","""NC_000001.11:155235217:C:G""","""1483G>C""","""Ala495Pro""","""CEGS2""","""CegsMutGDEh1042""","""F07""","""GDEhDisVCh_40054""","""G07""","""2""","""CegsMutGDDh1042""","""F07""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",1,1,1,1,0,null,null,0,null,null,0,null,null,"""GBA1_Ala495Pro""",…,368060,"""-""","""RCV000079341|RCV003129772""","""MedGen:CN169374|MedGen:C366190…","""not specified|not provided""","""germline""","""germline""","""GRCh38""","""NC_000001.11""","""1""",155235217,155235217,"""1q22""","""criteria provided, conflicting…",5,"""-""","""N""","""ClinGen:CA146980,UniProtKB:P04…",2,93450,155235217,"""C""","""G""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV002556287|SCV002756106|SCV0…","""-""","""-""","""1483G>C ""","""Ala495Pro""","""NM_000157.4""",1,"""3_Conflicting"""
"""GBA1""","""ENSG00000177628""",2,213515,"""CCSBVarC003870""","""ALE00003870""","""NC_000001.11:155238629:C:T""","""476G>A""","""Arg159Gln""","""CEGS2""","""CegsMutGDEh1046""","""F01""","""GDEhDisVCh_40054""","""E09""","""2""","""CegsMutGDDh1046""","""F01""","""NULL""","""NULL""","""NULL""","""NULL""","""NULL""",""

### 2.2 Merge the gnomad AF to VarChAMP alleles

In [14]:
varchamp_clinvar_gnomad = varchamp_clinvar.join(
    allele_freq_gnomad.select(pl.col(["varid","gnomad_af"])).lazy(),
    on="varid",how="left").collect()

## 3. Output the final results

In [15]:
varchamp_clinvar_gnomad.write_csv("../3_outputs/varchamp_seq_confirm_alleles_clinvar_gnomad.tsv", separator="\t")